# Set-up

In [ ]:
import os
import re
import time
import openpyxl
from datetime import datetime

from rag_agent.agent import RAGClient

from legal_rag.config import SUPREME_COURT_STATEMENT_FILE_STORE, GENERAL_LAW_FILE_STORE, SPECIFIC_LAW_FILE_STORE


from config import *

ImportError: cannot import name 'SPECIFIC_LAW_FILE_STORE' from 'legal_rag.config' (c:\Users\phurit.wara\OneDrive - Kiatnakin Phatra Bank Public Company Limited\RAG\legal_rag\config.py)

# Manage File Store

In [4]:
rag = RAGClient() 
rag.list_file_stores()

Legal Information (fileSearchStores/legal-information-iha7u4ocs45v) — 1 documents
  - The Civil and Commercial Code_20251111.pdf [fileSearchStores/legal-information-iha7u4ocs45v/documents/the-civil-and-commercial-co-bxga6ria2fwj]

SNS (fileSearchStores/sns-d4miedjtv6mf) — 1 documents
  - Analysis_of_Legal1.pdf [fileSearchStores/sns-d4miedjtv6mf/documents/analysisoflegal1pdf-bo1hpblnpapx]

L&C-Public Company Law Documents (fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5) — 4 documents
  - Public Company Law/Act_Public_Company_2535_20260204.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/public-company-lawactpublic-vvd67tmf5qs1]
  - General Law/OCPD_55.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/general-lawocpd55pdf-ng8uc3dcvx75]
  - General Law/OCPD_61.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/general-lawocpd61pdf-p7dd9edijwf0]
  - General Law/OCPD_65.pdf [fileSearchStores/lcpublic-company-law

[FileSearchStore(
   active_documents_count=1,
   create_time=datetime.datetime(2026, 1, 26, 11, 33, 39, 593478, tzinfo=TzInfo(0)),
   display_name='Legal Information',
   name='fileSearchStores/legal-information-iha7u4ocs45v',
   size_bytes=2638550,
   update_time=datetime.datetime(2026, 1, 26, 11, 33, 39, 593478, tzinfo=TzInfo(0))
 ),
 FileSearchStore(
   active_documents_count=1,
   create_time=datetime.datetime(2026, 2, 4, 8, 22, 38, 504510, tzinfo=TzInfo(0)),
   display_name='SNS',
   name='fileSearchStores/sns-d4miedjtv6mf',
   size_bytes=496345,
   update_time=datetime.datetime(2026, 2, 4, 8, 22, 38, 504510, tzinfo=TzInfo(0))
 ),
 FileSearchStore(
   active_documents_count=4,
   create_time=datetime.datetime(2026, 2, 9, 15, 1, 8, 600032, tzinfo=TzInfo(0)),
   display_name='L&C-Public Company Law Documents',
   name='fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5',
   size_bytes=1335768,
   update_time=datetime.datetime(2026, 2, 9, 15, 1, 8, 600032, tzinfo=TzInfo(0))
 

## Delete File Store

In [ ]:
# rag.delete_file_store('fileSearchStores/lcgeneral-law-documents-5ued446cduu7')

## Delete Documents

In [ ]:
# rag.delete_document("fileSearchStores/lcgeneral-law-documents-8uyfomy1ah0b/documents/public-company-lawactpublic-v0xozj9qf036")

## Create File Search Store

In [ ]:
# rag.create_file_store("L&C-Specific Law Documents")

Created store: L&C-Specific Law Documents (fileSearchStores/lcspecific-law-documents-chja6cb21834)


FileSearchStore(
  create_time=datetime.datetime(2026, 3, 4, 8, 47, 21, 816349, tzinfo=TzInfo(0)),
  display_name='L&C-Specific Law Documents',
  name='fileSearchStores/lcspecific-law-documents-chja6cb21834',
  update_time=datetime.datetime(2026, 3, 4, 8, 47, 21, 816349, tzinfo=TzInfo(0))
)

## Add File To File Search Store

### Flow 1: Folder-based upload (category = folder name)

In [ ]:
# Walk through sub-folders and upload each .docx file
base_path = './documents/KKP/LNC/Supreme Court Statement'
for category in sorted(os.listdir(base_path)):
    category_path = os.path.join(base_path, category)
    if not os.path.isdir(category_path):
        continue

    for fname in sorted(os.listdir(category_path)):
        if not fname.endswith('.docx'):
            continue

        file_rel_path = os.path.join('Supreme Court Statement', category, fname)
        metadata = [
            {'key': 'law_level', 'string_value': 'คำพิพากษาศาลฎีกา'},
            {'key': 'law_type', 'string_value': 'คำพิพากษาศาลฎีกา'},
            {'key': 'legal_category', 'string_value': category},
        ]

        operation = rag.store_file(file_rel_path, SUPREME_COURT_STATEMENT_FILE_STORE, metadata=metadata)

        while not operation.done:
            time.sleep(5)
            operation = rag.client.operations.get(operation)

        print(f"  ✓ Uploaded: {fname}")

print(f"\nDone! All documents uploaded to {SUPREME_COURT_STATEMENT_FILE_STORE}")

# Verify
docs = rag.list_documents(parent=SUPREME_COURT_STATEMENT_FILE_STORE)
print(f"\nDocuments in store ({len(list(docs))}):")
for doc in docs:
    print(f"  - {doc.display_name}")

### Flow 2: Excel-based upload (category = tag keywords from xlsx)

In [5]:
# ── 1. Read Excel: statement ID → tag mapping ────────────────────────
xlsx_path = os.path.join(DOCUMENTS_PATH, "Supreme Court Statement", "เลขฎีกาและ Tag key word.xlsx")
wb = openpyxl.load_workbook(xlsx_path)
ws = wb.active

id_to_tag = {}
for row in ws.iter_rows(min_row=2, values_only=True):
    statement_id, tag = row[0], row[1]
    if statement_id and tag:
        id_to_tag[str(statement_id).strip()] = str(tag).strip()

print(f"Loaded {len(id_to_tag)} statement-tag mappings from Excel")
for sid, tag in id_to_tag.items():
    print(f"  {sid} → {tag}")

# ── 2. Build file lookup: scan .docx files and extract IDs ───────────
base_path = os.path.join(DOCUMENTS_PATH, "Supreme Court Statement")
file_lookup = {}  # statement_id (num/num) → (category_folder, filename)

for category in sorted(os.listdir(base_path)):
    category_path = os.path.join(base_path, category)
    if not os.path.isdir(category_path):
        continue
    for fname in sorted(os.listdir(category_path)):
        if not fname.endswith('.docx') or fname.startswith('~$'):
            continue
        # Extract ID: "คำพิพากษาฎีกา 1140-2559.docx" → "1140/2559"
        # Also handles: "คำพิพากษาศาลฎีกาที่ 728_2545.docx" → "728/2545"
        match = re.search(r'(\d+)[-_](\d+)\.docx$', fname)
        if match:
            extracted_id = f"{match.group(1)}/{match.group(2)}"
            file_lookup[extracted_id] = (category, fname)

print(f"\nFound {len(file_lookup)} .docx files")

# ── 3. Match and upload ──────────────────────────────────────────────
uploaded = 0
skipped = []

for statement_id, tag in id_to_tag.items():
    if statement_id not in file_lookup:
        skipped.append(statement_id)
        print(f"  SKIP: No file found for ID {statement_id}")
        continue

    category, fname = file_lookup[statement_id]
    file_rel_path = os.path.join("Supreme Court Statement", category, fname)
    print(f"Uploading: {fname} (ID: {statement_id}, tag: {tag})")
    metadata = [
        {'key': 'law_level', 'string_value': 'คำพิพากษาศาลฎีกา'},
        {'key': 'law_type', 'string_value': 'คำพิพากษาศาลฎีกา'},
        {'key': 'legal_category', 'string_value': tag},
    ]

    operation = rag.store_file(fname, 
                               SUPREME_COURT_STATEMENT_FILE_STORE,
                               file_path=os.path.join(DOCUMENTS_PATH, "Supreme Court Statement", category),
                               metadata=metadata)

    while not operation.done:
        time.sleep(5)
        operation = rag.client.operations.get(operation)

    uploaded += 1
    print(f"  [{uploaded}/{len(id_to_tag)}] Uploaded: {fname} (tag: {tag})")

print(f"\nDone! Uploaded {uploaded} documents, skipped {len(skipped)}")
if skipped:
    print(f"Skipped IDs: {skipped}")

Loaded 20 statement-tag mappings from Excel
  728/2545 → จำนอง โอนทรัพย์สินที่จำนอง รับภาระจำนอง บุริมสิทธิ
  853/2567 → จำนอง โอนสิทธิเรียกร้อง บุริมสิทธิ คุ้มครองสิทธิ
  8085/2567 → จำนอง โอนทรัพย์สิน บุริมสิทธิ เพิกถอนการฉ้อฉล ดำเนินธุรกิจตามปกติในทางการค้า ล้มละลาย
  6008/2564 → จำนอง รับโอนที่ดิน ขายทอดตลาด เพิกถอนนิติกรรมการโอน บุริมสิทธิจำนอง ติดภาระจำนอง
  4457/2565 → โอนกรรมสิทธิ์ห้องชุด บุริมสิทธิ จำนอง ขายทอดตลาด ค่าใช้จ่ายส่วนกลาง
  813/2565 → เงินได้พึงประเมิน ภาษี บริษัทต่างประเทศ เครื่องหมายการค้า อนุสัญญาภาษีซ้อน
  5435/2564 → แผงค้า โอนสิทธิการเช่า ทรัพย์สิน สิทธิเรียกร้อง
  2297/2565 → สัญญาเช่า เช่าช่วง ละเมิด บริวาร ปฏิเสธคำฟ้อง
  12977/2558 → เช่าซื้อ ขาย บอกกล่าวการโอน ความยินยอมเป็นหนังสือ ยกข้อต่อสู้ ปัญหาข้อกฎหมายอันเกี่ยวด้วยความสงบเรียบร้อยของประชาชน
  1140/2559 → หย่า บันทึกแบ่งสินสมรส สินสมรส โอนสิทธิ ความยินยอม
  852/2555 → ปรับปรุงโครงสร้างหนี้ แปลงหนี้ใหม่ ผ่อนชำระหนี้ ดอกเบี้ยผิดนัด ค้ำประกัน ลูกหนี้ร่วม ผ่อนเวลา
  5732/2567 → ขยายระยะเวลาชำระหนี้ ผ่อนเ

UnicodeEncodeError: 'ascii' codec can't encode characters in position 0-12: ordinal not in range(128)

In [ ]:
# Verify uploads
docs = list(rag.client.file_search_stores.documents.list(parent=SUPREME_COURT_STATEMENT_FILE_STORE))
print(f"Total documents in store: {len(docs)}")
for doc in docs:
    print(f"  - {doc.display_name}")

In [10]:
# file_list = [
#   #  'The Civil and Commercial Code_20251111.pdf'
#   # '20240702_BOT_Policy.pdf',
#   # 'Analysis_of_Legal1.pdf'

#   "General Law/Act_Bankruptcy_20260204.pdf",
#   "General Law/Act_Business_Collateral_20260204.pdf",
#   "General Law/CCC_20251111.pdf",


#   ]

file_list = {
   "general_law": {
      
  # "General Law/Act_Bankruptcy_20260204.pdf": [
  #     {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
  #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
  #     {'key': 'law_name', 'string_value': 'พระราชบัญญัติล้มละลาย พ.ศ. 2483'},
  #     {'key': 'year', 'numeric_value': 2483 - 543},
  #     {'key': 'publish_date', 'numeric_value': int(datetime(2483 - 543, 12, 30).timestamp())},

     
  # ],
  # "General Law/Act_Business_Collateral_20260202.pdf": [
  #     {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
  #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
  #     {'key': 'law_name', 'string_value': 'พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558'},
  #     {'key': 'year', 'numeric_value': 2558 - 543},
  #     {'key': 'publish_date', 'numeric_value': int(datetime(2558 - 543, 10, 31).timestamp())},
     
  # ],
  # "General Law/CCC_20251111.pdf": [
  #     {'key': 'law_level', 'string_value': 'ประมวลกฎหมาย'},
  #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
  #     {'key': 'law_name', 'string_value': 'ประมวลกฎหมายแพ่งและพาณิชย์บรรพ 1 และ 2'},
  #     {'key': 'year', 'numeric_value': 2535 - 543},
  #     {'key': 'publish_date', 'numeric_value': int(datetime(2535 - 543, 3, 31).timestamp())},
     
  # ],

  # "General Law/Act_Building_1979.pdf": [
  #     {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
  #     {'key': 'law_type', 'string_value': 'กฎหมายทั่วไป'},
  #     {'key': 'law_name', 'string_value': 'พระราชบัญญัติควบคุมอาคาร พ.ศ. 2522'},
  #     {'key': 'year', 'numeric_value': 2522 - 543},
  #     {'key': 'publish_date', 'numeric_value': int(datetime(2522 - 543, 5, 8).timestamp())},
     
  # ],

  },
  "specific_law": {
    "Specific Law/Act_Public_Company_2535_20260204.pdf": [
      {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
      {'key': 'law_type', 'string_value': 'พระราชบัญญัติบริษัทมหาชนจำกัด กรณีลูกหนี้เป็นบริษัทมหาชน'},
      {'key': 'law_name', 'string_value': 'พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535'},
      {'key': 'year', 'numeric_value': 2535 - 543},
      {'key': 'publish_date', 'numeric_value': int(datetime(2535 - 543, 3, 29).timestamp())},
    ],
    
    "Specific Law/OCPD_55.pdf": [
      {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
      {'key': 'law_type', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค'},
      {'key': 'law_name', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค พ.ศ. 2555'},
      {'key': 'year', 'numeric_value': 2555 - 543},
      {'key': 'publish_date', 'numeric_value': int(datetime(2555 - 543, 11, 19).timestamp())},
    ],
    
    "Specific Law/OCPD_61.pdf": [
      {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
      {'key': 'law_type', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค'},
      {'key': 'law_name', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค พ.ศ. 2561'},
      {'key': 'year', 'numeric_value': 2561 - 543},
      {'key': 'publish_date', 'numeric_value': int(datetime(2561 - 543, 2, 16).timestamp())},
    ],
    
    "Specific Law/OCPD_65.pdf": [
      {'key': 'law_level', 'string_value': 'พระราชบัญญัติ'},
      {'key': 'law_type', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค'},
      {'key': 'law_name', 'string_value': 'พระราชบัญญัติคุ้มครองผู้บริโภค พ.ศ. 2565'},
      {'key': 'year', 'numeric_value': 2565 - 543},
      {'key': 'publish_date', 'numeric_value': int(datetime(2565 - 543, 10, 12).timestamp())},
    ]
    
  }
}

for file_name, metadata in file_list['specific_law'].items():
  operation = rag.store_file(file_name, SPECIFIC_LAW_FILE_STORE, metadata=metadata)

  while not operation.done:
      time.sleep(5)
      operation = rag.client.operations.get(operation)


NameError: name 'SPECIFIC_LAW_FILE_STORE' is not defined

In [ ]:
rag.list_documents(SPECIFIC_LAW_FILE_STORE)

  - Public Company Law/Act_Public_Company_2535_20260204.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/public-company-lawactpublic-vvd67tmf5qs1]
  - General Law/OCPD_55.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/general-lawocpd55pdf-ng8uc3dcvx75]
  - General Law/OCPD_61.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/general-lawocpd61pdf-p7dd9edijwf0]
  - General Law/OCPD_65.pdf [fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/general-lawocpd65pdf-2rbh2pnyy4ka]


[Document(
   create_time=datetime.datetime(2026, 2, 9, 15, 48, 55, 597313, tzinfo=TzInfo(0)),
   custom_metadata=[
     CustomMetadata(
       key='law_level',
       string_value='พระราชบัญญัติ'
     ),
     CustomMetadata(
       key='law_type',
       string_value='พรบ.บริษัทมหาชนจำกัด กรณีลูกหนี้เป็นบริษัทมหาชน'
     ),
     CustomMetadata(
       key='law_name',
       string_value='พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535'
     ),
     CustomMetadata(
       key='year',
       numeric_value=1992.0
     ),
     CustomMetadata(
       key='publish_date',
       numeric_value=701802000.0
     ),
   ],
   display_name='Public Company Law/Act_Public_Company_2535_20260204.pdf',
   mime_type='application/pdf',
   name='fileSearchStores/lcpublic-company-law-docume-x3nrtyfad1z5/documents/public-company-lawactpublic-vvd67tmf5qs1',
   size_bytes=660021,
   state=<DocumentState.STATE_ACTIVE: 'STATE_ACTIVE'>,
   update_time=datetime.datetime(2026, 2, 9, 15, 48, 59, 322719, tzinfo=TzInfo(0))
 